# Background

As part of our audit of the [Home Credit - Credit Risk Model Stability](https://www.kaggle.com/competitions/home-credit-credit-risk-model-stability) ADS, this is our EDA of the 16 train files from `train_credit_bureau_a_2_7` to `train_tax_registry_c_1` (bottom half of the dataset)

Steps:

1. **Types**: We cast every column by its suffix (P, A, D, M) and profile it. The cast rules follow [`Pipeline.set_table_dtypes`](https://www.kaggle.com/code/yuuniekiri/fork-of-home-credit-risk-lightgbm) in the audited code.
2. **Per-table checks**: We check that all keys are unique, raw date formats match, date and numeric ranges are correct, columns have proper cardinality, and inspect the most frequent values of masked columns.
3. **Missingness trends** per application.

# 0a. Setup

In [1]:
!pip install -q polars==2.1.0 kaggle==1.7.4.5

# 0b. Train Files

In [4]:
# Enabling Kaggle access
import os, json, subprocess
from pathlib import Path
for v in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
    os.environ.pop(v, None)
cred = Path("~/.kaggle/kaggle.json").expanduser()
cred.parent.mkdir(exist_ok=True)
try:
    from google.colab import userdata
    user, key = userdata.get("KAGGLE_USERNAME").strip(), userdata.get("KAGGLE_KEY").strip()
    cred.write_text(json.dumps({"username": user, "key": key}))
    print("using Colab secrets for user", user)
except Exception as e:
    print("Colab secrets not available:", type(e).__name__, "- upload kaggle.json instead")
    from google.colab import files
    up = files.upload()
    cred.write_bytes(up["kaggle.json"])
os.chmod(cred, 0o600)

COMP = "home-credit-credit-risk-model-stability"
check = subprocess.run(["kaggle", "competitions", "files", "-c", COMP], capture_output=True, text=True)
if check.returncode != 0:
    raise RuntimeError("Kaggle login or competition access failed:\n" + check.stdout + check.stderr +
                       "\n401 = wrong username/key; 403 = accept the competition rules on Kaggle first.")
print("Kaggle access OK")

using Colab secrets for user jleeman
Kaggle access OK


In [7]:
# Downloading training files
WORK = "/content/hc"
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
if not Path("parquet_files/train/train_base.parquet").exists():
    import zipfile
    zip_path = Path(f"/content/{COMP}.zip")
    if not (zip_path.exists() and zipfile.is_zipfile(zip_path)):
        zip_path.unlink(missing_ok=True)  # a partial zip makes kaggle 1.7.4.5 resume, and its resume crashes
        !kaggle competitions download -c {COMP} -p /content --force
    !unzip -q -o /content/{COMP}.zip "parquet_files/train/*" feature_definitions.csv -d {WORK}
n_files = len(list(Path("parquet_files/train").glob("*.parquet")))
assert n_files == 32, f"expected 32 train files, found {n_files}: check the download output above"
print(n_files, "train files")

/content/hc
100% 3.14G/3.14G [00:46<00:00, 57.4MB/s]
100% 3.14G/3.14G [00:46<00:00, 72.8MB/s]
32 train files


In [8]:
# Pull audited source code
!kaggle kernels pull yuuniekiri/fork-of-home-credit-risk-lightgbm -p fork_source

Source code downloaded to fork_source/fork-of-home-credit-risk-lightgbm.py


# 1. Type casting

In [10]:
"""
Type cast the bottom HALF (alphabetical) of the Home Credit train parquet files.

Per file: standardize dtypes (by the competition's column-suffix convention),
write a cleaned parquet, and write a column profile (dtype, null rate, distinct
values, min/max)

Kaggle competition data page (column suffix conventions);
dtype rules follow Pipeline.set_table_dtypes in
https://www.kaggle.com/code/yuuniekiri/fork-of-home-credit-risk-lightgbm

Memory: casting is written with a streaming sink and profiling runs on a few
columns at a time to avoid crashing.
"""
from pathlib import Path
import polars as pl

ROOT = Path(".")
TRAIN_DIR = ROOT / "parquet_files" / "train"
CLEAN_DIR = ROOT / "work" / "clean"
PROFILE_DIR = ROOT / "work" / "profiles"
COMPRESSION = "zstd"
COLS_PER_PASS = 25

for d in (CLEAN_DIR, PROFILE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- 1. choose the files ----
files = sorted(p for p in TRAIN_DIR.glob("*.parquet"))   # alphabetical by filename
if not files:
    raise SystemExit(f"No .parquet files found in {TRAIN_DIR.resolve()}. Run this from the folder that contains parquet_files/, or fix ROOT.")
mid = len(files) // 2
(PROFILE_DIR / f"file_list_bottom_half.txt").write_text("\n".join(p.name for p in files[mid:]))
print(f"{len(files)} train files; bottom half = {len(files[mid:])} files:")
for p in files[mid:]:
    print("  ", p.name)

# ---- 2. dtype rules (suffix convention) ----
ID_COLS = {"case_id", "WEEK_NUM", "num_group1", "num_group2"}

def casts_for(schema):
    """Return (cast expressions, names of date columns that were stored as text)."""
    exprs, text_dates = [], []
    for c, dtype in schema.items():
        if c in ID_COLS:
            exprs.append(pl.col(c).cast(pl.Int64))
        elif c == "date_decision" or c[-1] == "D":
            if dtype == pl.String:
                exprs.append(pl.col(c).str.to_date(strict=False))
                text_dates.append(c)
            else:
                exprs.append(pl.col(c).cast(pl.Date))
        elif c[-1] in ("P", "A"):
            exprs.append(pl.col(c).cast(pl.Float64))
        elif c[-1] == "M":
            exprs.append(pl.col(c).cast(pl.String))
        # "L" and "T" columns are left as stored
    return exprs, text_dates

# ---- 3. profile a cleaned file a few columns at a time ----
def profile(path):
    lf = pl.scan_parquet(path)
    schema = lf.collect_schema()
    n = lf.select(pl.len()).collect().item()
    rows = []
    cols = list(schema.names())
    for i in range(0, len(cols), COLS_PER_PASS):
        batch = cols[i:i + COLS_PER_PASS]
        aggs = []
        for c in batch:
            aggs += [pl.col(c).null_count().alias(f"{c}||nulls"),
                     pl.col(c).drop_nulls().n_unique().alias(f"{c}||distinct")]
            if schema[c].is_numeric() or schema[c] == pl.Date:
                aggs += [pl.col(c).min().alias(f"{c}||min"), pl.col(c).max().alias(f"{c}||max")]
        res = lf.select(aggs).collect().row(0, named=True)
        for c in batch:
            rows.append({
                "column": c, "dtype": str(schema[c]), "rows": n,
                "null_rate": res[f"{c}||nulls"] / n if n else None,
                "distinct": res[f"{c}||distinct"],
                "min": None if res.get(f"{c}||min") is None else str(res[f"{c}||min"]),
                "max": None if res.get(f"{c}||max") is None else str(res[f"{c}||max"]),
            })
    return pl.DataFrame(rows, schema_overrides={"min": pl.String, "max": pl.String}, infer_schema_length=None)

# ---- 4. clean each file (resumable) ----
for p in files[mid:]:
    out = CLEAN_DIR / p.name
    prof_path = PROFILE_DIR / f"{p.stem}_profile.csv"
    if out.exists() and prof_path.exists():
        print(f"skip (already done): {p.name}")
        continue
    tmp = out.with_suffix(".tmp")
    src = pl.scan_parquet(p)
    exprs, text_dates = casts_for(src.collect_schema())
    lf = src.with_columns(exprs)
    lf.sink_parquet(tmp, compression=COMPRESSION)
    tmp.replace(out)
    try:
        prof = profile(out)
    except Exception as e:
        out.unlink(missing_ok=True)
        print(f"\nFAILED read-back of {p.name}: {e}\nDeleted the bad output. Check disk space (df -h .) and the Polars/CPU warning, "
              f"or set COMPRESSION = 'snappy' and rerun (finished files are skipped).")
        raise
    prof.write_csv(prof_path)
    if text_dates:
        before = src.select([pl.col(c).null_count() for c in text_dates]).collect().row(0)
        after = pl.scan_parquet(out).select([pl.col(c).null_count() for c in text_dates]).collect().row(0)
        for c, b, a in zip(text_dates, before, after):
            msg = f"{p.name}: text date column {c}: nulls {b} -> {a}" + ("  <-- UNPARSED VALUES" if a > b else "")
            print("  ", msg)
            with open(PROFILE_DIR / "date_parse_notes.txt", "a") as fh:
                fh.write(msg + "\n")
    print(f"cleaned {p.name}: {prof['rows'][0]} rows, {prof.height} columns")

print("done. cleaned files in", CLEAN_DIR, "; profiles in", PROFILE_DIR)

32 train files; bottom half = 16 files:
   train_credit_bureau_a_2_7.parquet
   train_credit_bureau_a_2_8.parquet
   train_credit_bureau_a_2_9.parquet
   train_credit_bureau_b_1.parquet
   train_credit_bureau_b_2.parquet
   train_debitcard_1.parquet
   train_deposit_1.parquet
   train_other_1.parquet
   train_person_1.parquet
   train_person_2.parquet
   train_static_0_0.parquet
   train_static_0_1.parquet
   train_static_cb_0.parquet
   train_tax_registry_a_1.parquet
   train_tax_registry_b_1.parquet
   train_tax_registry_c_1.parquet
skip (already done): train_credit_bureau_a_2_7.parquet
skip (already done): train_credit_bureau_a_2_8.parquet
skip (already done): train_credit_bureau_a_2_9.parquet
skip (already done): train_credit_bureau_b_1.parquet
skip (already done): train_credit_bureau_b_2.parquet
skip (already done): train_debitcard_1.parquet
skip (already done): train_deposit_1.parquet
skip (already done): train_other_1.parquet
skip (already done): train_person_1.parquet
skip (alr

We set column types using the competition's suffix convention, following the fork's set_table_dtypes. All text date columns parsed without loss: null counts were unchanged for every one, so typing introduced no new missing values. No values were changed in this step.

The next cell outputs a condensed summmary of all 16 tables and an example of a full column-by-column profile (`person_1`)

In [12]:
KEYS = {"case_id", "WEEK_NUM", "num_group1", "num_group2"}
prof = pl.concat([pl.read_csv(f, infer_schema_length=None).with_columns(pl.lit(f.stem.removesuffix("_profile")).alias("table"))
                  for f in sorted(PROFILE_DIR.glob("train_*_profile.csv"))], how="diagonal_relaxed")
prof = prof.filter(pl.col("table").is_in([p.stem for p in mine]) & ~pl.col("column").is_in(list(KEYS)))
summary = (prof.group_by("table", maintain_order=True)
           .agg(pl.col("rows").first(), pl.len().alias("data_columns"),
                pl.col("null_rate").mean().round(3).alias("mean_null_rate"),
                (pl.col("null_rate") > 0.5).sum().alias("cols_over_50pct_null"),
                (pl.col("null_rate") > 0.95).sum().alias("cols_over_95pct_null"),
                (pl.col("distinct") <= 1).sum().alias("cols_constant_or_empty"))
           .sort("table"))
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=200):
    print("Condensed summary of all 16 tables")
    print(summary)
    print("\nExample, person_1 (one row per column):")
    print(prof.filter(pl.col("table") == "train_person_1").select("column", "dtype", "null_rate", "distinct", "min", "max")
          .with_columns(pl.col("null_rate").round(3)))

Condensed summary of all 16 tables
shape: (16, 7)
┌───────────────────────────┬──────────┬──────────────┬────────────────┬──────────────────────┬──────────────────────┬────────────────────────┐
│ table                     ┆ rows     ┆ data_columns ┆ mean_null_rate ┆ cols_over_50pct_null ┆ cols_over_95pct_null ┆ cols_constant_or_empty │
│ ---                       ┆ ---      ┆ ---          ┆ ---            ┆ ---                  ┆ ---                  ┆ ---                    │
│ str                       ┆ i64      ┆ u32          ┆ f64            ┆ u32                  ┆ u32                  ┆ u32                    │
╞═══════════════════════════╪══════════╪══════════════╪════════════════╪══════════════════════╪══════════════════════╪════════════════════════╡
│ train_credit_bureau_a_2_7 ┆ 8055986  ┆ 16           ┆ 0.397          ┆ 8                    ┆ 2                    ┆ 0                      │
│ train_credit_bureau_a_2_8 ┆ 13927071 ┆ 16           ┆ 0.398          ┆ 8            

**Note:** Null rates in this table count real nulls only. Masked columns show 0% because missing values are coded as `a55475b1`; we measure the true missing rate later.

# 2. Per-table checks

This is a setup cell that will run once and then can be called on each table table with `analyze(...)`

In [32]:
import html
from pathlib import Path
import polars as pl

RAW = Path("parquet_files/train")
CLEAN = Path("work/clean")
OUT = Path("work/clean2"); OUT.mkdir(parents=True, exist_ok=True)
NOTES = Path("work/notes"); NOTES.mkdir(parents=True, exist_ok=True)

pl.Config.set_tbl_rows(40); pl.Config.set_tbl_cols(14); pl.Config.set_fmt_str_lengths(50)

KEY_COLS = {"case_id", "WEEK_NUM", "num_group1", "num_group2"}
BATCH = 15

def lz(name, raw=False):
    return pl.scan_parquet((RAW if raw else CLEAN) / f"{name}.parquet")

def batches(cols, n=BATCH):
    for i in range(0, len(cols), n):
        yield cols[i:i + n]

try:
    from IPython.display import display, HTML
    IN_NOTEBOOK = get_ipython() is not None
except (ImportError, NameError):
    IN_NOTEBOOK = False

def show(title, df, rows=200, expanded=False):
    """In a notebook each section is a collapsible block: click the arrow to open it."""
    if df is None or df.height == 0:
        print(f"--- {title}: nothing to report"); return
    with pl.Config(tbl_rows=rows, tbl_cols=-1, fmt_str_lengths=60, tbl_width_chars=250):
        if not IN_NOTEBOOK:
            print(f"\n--- {title} ---"); print(df); return
        display(HTML(f"<details{' open' if expanded else ''}><summary><b>{html.escape(title)}</b> "
                     f"({df.height:,} rows)</summary>{df._repr_html_()}</details>"))

# --- 1. keys -----------------------------------------------------------------
def key_check(lf, keys):
    n = lf.select(pl.len()).collect().item()
    u = lf.select(pl.struct(keys).n_unique()).collect().item()
    return pl.DataFrame({"keys": [", ".join(keys)], "rows": [n], "unique_keys": [u], "duplicate_rows": [n - u]})

# --- 2. cardinality of string / boolean columns --------------------------------
def cardinality(lf, cols=None):
    sch = lf.collect_schema()
    cols = cols or [c for c, t in sch.items() if t in (pl.String, pl.Boolean) and c not in KEY_COLS]
    rows = []
    for b in batches(cols):
        r = lf.select([pl.col(c).drop_nulls().n_unique().alias(c) for c in b]).collect().row(0, named=True)
        rows += [{"column": c, "distinct": r[c]} for c in b]
    return pl.DataFrame(rows).sort("distinct", descending=True) if rows else None

# --- 3. top values (use on M columns to find placeholder categories) -----------
def top_values(lf, cols, top=3):
    n = lf.select(pl.len()).collect().item()
    parts = []
    for c in cols:
        d = (lf.group_by(c).len().sort("len", descending=True).head(top).collect()
             .select(pl.lit(c).alias("column"), pl.col(c).cast(pl.String).alias("value"),
                     pl.col("len").alias("rows"), (pl.col("len") / n).round(4).alias("share")))
        parts.append(d)
    return pl.concat(parts) if parts else None

# --- 4. raw date formats (text date columns) ----------------------------------
def date_formats(raw_lf, cols=None):
    sch = raw_lf.collect_schema()
    cols = cols or [c for c, t in sch.items() if (c.endswith("D") or c == "date_decision") and t == pl.String]
    parts = []
    for c in cols:
        d = (raw_lf.select(pl.col(c).drop_nulls().str.replace_all(r"\d", "9").alias("pattern"))
             .group_by("pattern").len().sort("len", descending=True).head(5).collect()
             .with_columns(pl.lit(c).alias("column")).select("column", "pattern", pl.col("len").alias("rows")))
        parts.append(d)
    return pl.concat(parts) if parts else None

# --- 5. date ranges -------------------------------------------------------------
def date_ranges(lf, lo_year=1930, hi_year=2030):
    sch = lf.collect_schema()
    cols = [c for c, t in sch.items() if t == pl.Date]
    rows = []
    for b in batches(cols):
        aggs = []
        for c in b:
            aggs += [pl.col(c).min().alias(f"{c}||min"), pl.col(c).max().alias(f"{c}||max"),
                     (pl.col(c).dt.year() < lo_year).sum().alias(f"{c}||early"),
                     (pl.col(c).dt.year() > hi_year).sum().alias(f"{c}||late"),
                     (pl.col(c).dt.day() == 1).mean().alias(f"{c}||day1"),
                     pl.col(c).is_null().mean().alias(f"{c}||null")]
        r = lf.select(aggs).collect().row(0, named=True)
        for c in b:
            rows.append({"column": c, "min": str(r[f"{c}||min"]), "max": str(r[f"{c}||max"]),
                         f"before_{lo_year}": r[f"{c}||early"], f"after_{hi_year}": r[f"{c}||late"],
                         "share_day_is_1": round(r[f"{c}||day1"] or 0, 3), "null_rate": round(r[f"{c}||null"], 3)})
    return pl.DataFrame(rows) if rows else None

# --- 6. numeric ranges ----------------------------------------------------------
def numeric_summary(lf):
    sch = lf.collect_schema()
    cols = [c for c, t in sch.items() if t.is_numeric() and c not in KEY_COLS]
    rows = []
    for b in batches(cols, 10):
        aggs = []
        for c in b:
            x = pl.col(c)
            aggs += [x.min().alias(f"{c}||min"), x.quantile(0.01).alias(f"{c}||p1"), x.median().alias(f"{c}||p50"),
                     x.quantile(0.99).alias(f"{c}||p99"), x.max().alias(f"{c}||max"),
                     (x < 0).sum().alias(f"{c}||neg"), (x == 0).mean().alias(f"{c}||zero")]
        r = lf.select(aggs).collect().row(0, named=True)
        for c in b:
            rows.append({"column": c, "min": r[f"{c}||min"], "p1": r[f"{c}||p1"], "median": r[f"{c}||p50"],
                         "p99": r[f"{c}||p99"], "max": r[f"{c}||max"], "n_negative": r[f"{c}||neg"],
                         "share_zero": round(r[f"{c}||zero"] or 0, 3)})
    return pl.DataFrame(rows, infer_schema_length=None) if rows else None

# --- one call per table -----------------------------------------------------------
def analyze(name, keys, rows=200):
    lf, raw = lz(name), lz(name, raw=True)
    sch = lf.collect_schema()
    n = lf.select(pl.len()).collect().item()
    print(f"\n=================== {name}: {n:,} rows, {len(sch)} columns ===================")
    show("keys", key_check(lf, keys))
    show("text date formats (pattern, 9 = digit)", date_formats(raw))
    show("date ranges", date_ranges(lf), rows)
    show("cardinality of string/boolean columns", cardinality(lf), rows)
    mcols = [c for c in sch if c.endswith("M")]
    show("top values of masked (M) columns; a55475b1 is a mask for a missing value", top_values(lf, mcols), rows)
    show("numeric ranges", numeric_summary(lf), rows)

## Depth 0: one row per case

In [18]:
analyze("train_static_0_0", ["case_id"])


=================== train_static_0_0: 1,003,757 rows, 168 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id""",1003757,1003757,0


column,pattern,rows
str,str,u32
"""datefirstoffer_1144D""","""9999-99-99""",443044
"""datelastinstal40dpd_247D""","""9999-99-99""",71493
"""datelastunpaid_3546854D""","""9999-99-99""",397007
"""dtlastpmtallstes_4499206D""","""9999-99-99""",244853
"""firstclxcampaign_1125D""","""9999-99-99""",432102
"""firstdatedue_489D""","""9999-99-99""",664070
"""lastactivateddate_801D""","""9999-99-99""",682915
"""lastapplicationdate_877D""","""9999-99-99""",782997
"""lastapprdate_640D""","""9999-99-99""",690634


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""datefirstoffer_1144D""","""2007-04-30""","""2019-12-25""",0,0,0.023,0.559
"""datelastinstal40dpd_247D""","""2006-02-04""","""2019-11-17""",0,0,0.033,0.929
"""datelastunpaid_3546854D""","""2006-02-09""","""2019-12-29""",0,0,0.031,0.604
"""dtlastpmtallstes_4499206D""","""2006-03-10""","""2019-12-29""",0,0,0.028,0.756
"""firstclxcampaign_1125D""","""2016-01-31""","""2019-12-30""",0,0,0.046,0.57
"""firstdatedue_489D""","""2006-01-31""","""2020-01-30""",0,0,0.033,0.338
"""lastactivateddate_801D""","""2006-01-08""","""2019-12-30""",0,0,0.03,0.32
"""lastapplicationdate_877D""","""2006-01-03""","""2019-12-30""",0,0,0.035,0.22
"""lastapprdate_640D""","""2006-01-07""","""2019-12-30""",0,0,0.034,0.312


column,distinct
str,i64
"""previouscontdistrict_112M""",199
"""lastcancelreason_561M""",67
"""lastrejectcommoditycat_161M""",44
"""lastapprcommoditycat_1041M""",43
"""lastrejectreason_759M""",18
"""lastrejectreasonclient_4145040M""",11
"""lastst_736L""",11
"""credtype_322L""",3
"""disbursementtype_67L""",3


column,value,rows,share
str,str,u32,f64
"""lastapprcommoditycat_1041M""","""a55475b1""",517454,0.5155
"""lastapprcommoditycat_1041M""","""P159_130_59""",184373,0.1837
"""lastapprcommoditycat_1041M""","""P12_6_178""",112401,0.112
"""lastapprcommoditytypec_5251766M""","""a55475b1""",1003757,1.0
"""lastcancelreason_561M""","""a55475b1""",721831,0.7191
"""lastcancelreason_561M""","""P94_109_143""",188780,0.1881
"""lastcancelreason_561M""","""P85_114_140""",14833,0.0148
"""lastrejectcommoditycat_161M""","""a55475b1""",789334,0.7864
"""lastrejectcommoditycat_161M""","""P159_130_59""",71239,0.071


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""actualdpdtolerance_344P""",0.0,0.0,0.0,0.0,3676.0,0,0.996
"""amtinstpaidbefduel24m_4187115A""",0.0,0.0,26937.0,295557.44,992476.5,0,0.174
"""annuity_780A""",83.0,675.60004,3000.0,14248.4,106007.0,0,0.0
"""annuitynextmonth_57A""",0.0,0.0,0.0,11417.2,85620.805,0,0.703
"""applicationcnt_361L""",0.0,0.0,0.0,0.0,5.0,0,1.0
"""applications30d_658L""",0.0,0.0,0.0,2.0,25.0,0,0.901
"""applicationscnt_1086L""",0.0,0.0,0.0,12.0,443.0,0,0.944
"""applicationscnt_464L""",0.0,0.0,0.0,28.0,247.0,0,0.899
"""applicationscnt_629L""",0.0,0.0,0.0,7.0,77.0,0,0.94


`static_0_0` has exactly one row per application, and every text date uses one format. The most common value in several masked columns is the code `a55475b1`, which we will treat as missing. Negative values occur only in `days before due` columns, where they are valid, and in a few `outstanding_balance` columns with extreme outliers.

In [19]:
analyze("train_static_0_1", ["case_id"])


=================== train_static_0_1: 522,902 rows, 168 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id""",522902,522902,0


column,pattern,rows
str,str,u32
"""datefirstoffer_1144D""","""9999-99-99""",249880
"""datelastinstal40dpd_247D""","""9999-99-99""",62325
"""datelastunpaid_3546854D""","""9999-99-99""",241993
"""dtlastpmtallstes_4499206D""","""9999-99-99""",304687
"""firstclxcampaign_1125D""","""9999-99-99""",235343
"""firstdatedue_489D""","""9999-99-99""",380486
"""lastactivateddate_801D""","""9999-99-99""",390157
"""lastapplicationdate_877D""","""9999-99-99""",438525
"""lastapprdate_640D""","""9999-99-99""",393984


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""datefirstoffer_1144D""","""2007-04-30""","""2020-06-22""",0,0,0.022,0.522
"""datelastinstal40dpd_247D""","""2006-02-23""","""2020-09-06""",0,0,0.034,0.881
"""datelastunpaid_3546854D""","""2006-02-08""","""2020-10-18""",0,0,0.03,0.537
"""dtlastpmtallstes_4499206D""","""2006-02-22""","""2020-10-18""",0,0,0.031,0.417
"""firstclxcampaign_1125D""","""2016-01-31""","""2020-06-23""",0,0,0.046,0.55
"""firstdatedue_489D""","""2006-01-06""","""2020-11-11""",0,0,0.033,0.272
"""lastactivateddate_801D""","""2006-01-11""","""2020-10-19""",0,0,0.03,0.254
"""lastapplicationdate_877D""","""2006-01-10""","""2020-10-19""",0,0,0.035,0.161
"""lastapprdate_640D""","""2006-01-10""","""2020-10-19""",0,0,0.035,0.247


column,distinct
str,i64
"""previouscontdistrict_112M""",223
"""lastapprcommoditytypec_5251766M""",214
"""lastrejectcommodtypec_5251769M""",187
"""lastcancelreason_561M""",67
"""lastapprcommoditycat_1041M""",42
"""lastrejectcommoditycat_161M""",40
"""lastrejectreason_759M""",17
"""lastrejectreasonclient_4145040M""",14
"""lastst_736L""",11


column,value,rows,share
str,str,u32,f64
"""lastapprcommoditycat_1041M""","""a55475b1""",334792,0.6403
"""lastapprcommoditycat_1041M""","""P159_130_59""",72133,0.1379
"""lastapprcommoditycat_1041M""","""P12_6_178""",43656,0.0835
"""lastapprcommoditytypec_5251766M""","""a55475b1""",499900,0.956
"""lastapprcommoditytypec_5251766M""","""P75_90_70""",7154,0.0137
"""lastapprcommoditytypec_5251766M""","""P111_89_135""",2081,0.004
"""lastcancelreason_561M""","""a55475b1""",352222,0.6736
"""lastcancelreason_561M""","""P94_109_143""",112017,0.2142
"""lastcancelreason_561M""","""P85_114_140""",8316,0.0159


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""actualdpdtolerance_344P""",0.0,0.0,0.0,0.0,4206.0,0,0.997
"""amtinstpaidbefduel24m_4187115A""",0.0,0.0,34623.065,347329.3,1408010.2,0,0.157
"""annuity_780A""",80.8,774.0,3478.2,15266.2,91601.4,0,0.0
"""annuitynextmonth_57A""",0.0,0.0,0.0,12625.8,87500.0,0,0.676
"""applicationcnt_361L""",0.0,0.0,0.0,0.0,3.0,0,1.0
"""applications30d_658L""",0.0,0.0,0.0,2.0,28.0,0,0.885
"""applicationscnt_1086L""",0.0,0.0,0.0,13.0,728.0,0,0.954
"""applicationscnt_464L""",0.0,0.0,0.0,13.0,246.0,0,0.942
"""applicationscnt_629L""",0.0,0.0,0.0,3.0,90.0,0,0.967


`static_0_1` covers later applications (decided Dec 2019 to Oct 2020). Two commodity-type columns exist only in this period, and several columns are filled more often, so missingness in `static_0_0` partly reflects when an application was made.

The next cell highlights exactly where the two tables split by decision date. `static_0_0` holds applications from January 1 to December 16, 2019 and `static_0_1` holds applications from December 17, 2019 to October 5, 2020.

In [23]:
# decision dates per table: which period does each table cover?
DECISION = (pl.scan_parquet(RAW / "train_base.parquet")
            .select(pl.col("case_id").cast(pl.Int64), pl.col("date_decision").str.to_date().alias("decision")))

def period(name):
    c = lz(name).select("case_id").unique().join(DECISION, on="case_id")
    return c.select(pl.lit(name).alias("table"), pl.len().alias("applications"),
                    pl.col("decision").min().alias("first_decision"),
                    pl.col("decision").max().alias("last_decision")).collect()

show("applications and decision dates covered", pl.concat([period(f"train_static_0_{s}") for s in ("0", "1")]))

table,applications,first_decision,last_decision
str,u32,date,date
"""train_static_0_0""",1003757,2019-01-01,2019-12-16
"""train_static_0_1""",522902,2019-12-17,2020-10-05


In [24]:
analyze("train_static_cb_0", ["case_id"])


=================== train_static_cb_0: 1,500,476 rows, 53 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id""",1500476,1500476,0


column,pattern,rows
str,str,u32
"""assignmentdate_238D""","""9999-99-99""",136996
"""assignmentdate_4527235D""","""9999-99-99""",114978
"""assignmentdate_4955616D""","""9999-99-99""",71633
"""birthdate_574D""","""9999-99-99""",607871
"""dateofbirth_337D""","""9999-99-99""",1385691
"""dateofbirth_342D""","""9999-99-99""",36500
"""responsedate_1012D""","""9999-99-99""",720000
"""responsedate_4527233D""","""9999-99-99""",660327
"""responsedate_4917613D""","""9999-99-99""",224912


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""assignmentdate_238D""","""1974-05-30""","""2019-10-22""",0,0,0.032,0.909
"""assignmentdate_4527235D""","""2019-09-13""","""2020-10-19""",0,0,0.037,0.923
"""assignmentdate_4955616D""","""1984-12-05""","""2020-09-22""",0,0,0.031,0.952
"""birthdate_574D""","""1943-03-01""","""1998-09-01""",0,0,1.0,0.595
"""dateofbirth_337D""","""1900-01-01""","""2020-01-01""",24,0,1.0,0.076
"""dateofbirth_342D""","""1915-01-01""","""2015-09-01""",2,0,1.0,0.976
"""responsedate_1012D""","""2019-01-02""","""2019-10-22""",0,0,0.035,0.52
"""responsedate_4527233D""","""2019-09-13""","""2020-10-19""",0,0,0.036,0.56
"""responsedate_4917613D""","""2020-03-26""","""2020-10-19""",0,0,0.04,0.85


column,distinct
str,i64
"""riskassesment_302T""",16
"""maritalst_385M""",6
"""maritalst_893M""",6
"""education_1103M""",5
"""education_88M""",5
"""requesttype_4525192L""",3
"""description_5085714M""",2


column,value,rows,share
str,str,u32,f64
"""description_5085714M""","""a55475b1""",1316125,0.8771
"""description_5085714M""","""2fc785b2""",184351,0.1229
"""education_1103M""","""a55475b1""",859962,0.5731
"""education_1103M""","""6b2ae0fa""",452449,0.3015
"""education_1103M""","""717ddd49""",135342,0.0902
"""education_88M""","""a55475b1""",1484960,0.9897
"""education_88M""","""6b2ae0fa""",11673,0.0078
"""education_88M""","""717ddd49""",3280,0.0022
"""maritalst_385M""","""a55475b1""",661592,0.4409


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""contractssum_5085716L""",0.0,0.0,307282.4,4.7870e6,3.1297e7,0,0.114
"""days120_123L""",0.0,0.0,1.0,9.0,109.0,0,0.374
"""days180_256L""",0.0,0.0,2.0,13.0,110.0,0,0.274
"""days30_165L""",0.0,0.0,0.0,4.0,22.0,0,0.662
"""days360_512L""",0.0,0.0,3.0,24.0,115.0,0,0.137
"""days90_310L""",0.0,0.0,1.0,7.0,41.0,0,0.449
"""firstquarter_103L""",0.0,0.0,2.0,17.0,76.0,0,0.291
"""for3years_128L""",0.0,0.0,0.0,0.0,1.0,0,1.0
"""for3years_504L""",0.0,0.0,2.0,27.0,57.0,0,0.286


In [26]:
# static_cb_0 birth date vs person_1 birth date (consistency)
cb = lz("train_static_cb_0").select("case_id", "dateofbirth_337D", "birthdate_574D")
p1 = lz("train_person_1").filter(pl.col("num_group1") == 0).select("case_id", "birth_259D")
j = cb.join(p1, on="case_id", how="inner").collect()
both = j.filter(pl.col("dateofbirth_337D").is_not_null() & pl.col("birth_259D").is_not_null())
same = both.filter(pl.col("dateofbirth_337D") == pl.col("birth_259D")).height
sameym = both.filter((pl.col("dateofbirth_337D").dt.year() == pl.col("birth_259D").dt.year()) &
                     (pl.col("dateofbirth_337D").dt.month() == pl.col("birth_259D").dt.month())).height
print(f"cases with both: {both.height:,}; identical date: {same:,}; same year+month: {sameym:,}")
diff = both.filter(pl.col("dateofbirth_337D") != pl.col("birth_259D"))
odd = (pl.col("dateofbirth_337D").dt.year() < 1930) | (pl.col("dateofbirth_337D").dt.year() > 2005)
gap = ((pl.col("dateofbirth_337D") - pl.col("birth_259D")).dt.total_days().abs() / 365.25)
print(f"disagreements: {diff.height:,}, of which dateofbirth_337D outside 1930-2005: {diff.filter(odd).height:,}")

cases with both: 1,385,691; identical date: 1,383,624; same year+month: 1,383,624
disagreements: 2,067, of which dateofbirth_337D outside 1930-2005: 65


`static_cb_0` covers 98.3% of applications. All birth dates are rounded to the 1st of the month. `dateofbirth_337D` contains implausible values (outside 1930-2005), but agrees exactly with `birth_259D` in 99.85% of cases, so we can use `birth_259D` for age moving forward.

## Depth 1: several rows per case (key = case_id, num_group1)

In [27]:
analyze("train_person_1", ["case_id", "num_group1"])


=================== train_person_1: 2,973,991 rows, 37 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",2973991,2973991,0


column,pattern,rows
str,str,u32
"""birth_259D""","""9999-99-99""",1526659
"""birthdate_87D""","""9999-99-99""",24916
"""empl_employedfrom_271D""","""9999-99-99""",566701


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""birth_259D""","""1943-03-01""","""1999-10-01""",0,0,1.0,0.487
"""birthdate_87D""","""1911-12-01""","""2015-12-01""",2,0,1.0,0.992
"""empl_employedfrom_271D""","""1963-06-15""","""2020-09-15""",0,0,0.003,0.809


column,distinct
str,i64
"""registaddr_zipcode_184M""",3531
"""contaddr_zipcode_807M""",3530
"""empladdr_zipcode_114M""",3339
"""registaddr_district_1083M""",991
"""contaddr_district_15M""",975
"""empladdr_district_926M""",223
"""empl_industry_691L""",24
"""relationshiptoclient_415T""",10
"""relationshiptoclient_642T""",10


column,value,rows,share
str,str,u32,f64
"""contaddr_district_15M""","""a55475b1""",1449674,0.4875
"""contaddr_district_15M""","""P131_33_167""",80070,0.0269
"""contaddr_district_15M""","""P197_47_166""",77354,0.026
"""contaddr_zipcode_807M""","""a55475b1""",1474844,0.4959
"""contaddr_zipcode_807M""","""P161_14_174""",97924,0.0329
"""contaddr_zipcode_807M""","""P144_138_111""",90612,0.0305
"""education_927M""","""a55475b1""",2234573,0.7514
"""education_927M""","""P97_36_170""",415087,0.1396
"""education_927M""","""P33_146_175""",263189,0.0885


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""childnum_185L""",0.0,0.0,0.0,4.0,11.0,0,0.61
"""mainoccupationinc_384A""",0.0,14000.0,50000.0,190000.0,200000.0,0,0.0
"""personindex_1023L""",0.0,0.0,0.0,2.0,6.0,0,0.655
"""persontype_1072L""",1.0,1.0,1.0,5.0,5.0,0,0.0
"""persontype_792L""",1.0,1.0,1.0,5.0,5.0,0,0.0


In [28]:
# person_1: is sex/birth only on the applicant row?
p = lz("train_person_1")
by_grp = (p.group_by("num_group1").agg(pl.len().alias("rows"),
          pl.col("sex_738L").is_not_null().sum().alias("sex_738L_nonnull"),
          pl.col("birth_259D").is_not_null().sum().alias("birth_259D_nonnull"),
          pl.col("gender_992L").is_not_null().sum().alias("gender_992L_nonnull"))
          .sort("num_group1").collect())
show("non-null protected-attribute counts by num_group1 (0 = applicant)", by_grp)
rows_per_case = (p.group_by("case_id").agg(pl.len().alias("rows_per_case"))
                 .group_by("rows_per_case").agg(pl.len().alias("cases")).sort("rows_per_case").collect())
show("rows per case and how many cases have that many", rows_per_case)
agree = (p.filter(pl.col("sex_738L").is_not_null() & pl.col("gender_992L").is_not_null())
         .group_by("sex_738L", "gender_992L").len().collect())
show("sex_738L vs gender_992L where both present", agree)
show("gender_992L by row (0 = applicant)",
     p.filter(pl.col("gender_992L").is_not_null()).group_by("num_group1", "gender_992L").len()
     .sort("num_group1", "gender_992L").collect())
inc = pl.col("mainoccupationinc_384A")
show("rounding and caps", p.select(
    (pl.col("empl_employedfrom_271D").dt.day() == 15).mean().round(3).alias("employment_start_on_15th"),
    (pl.col("birth_259D").dt.day() == 1).mean().round(3).alias("birth_date_on_1st"),
    (inc == 200000).sum().alias("income_exactly_200000"),
    inc.is_not_null().sum().alias("income_non_null")).collect())

num_group1,rows,sex_738L_nonnull,birth_259D_nonnull,gender_992L_nonnull
i64,u32,u32,u32,u32
0,1526659,1526659,1526659,12458
1,757320,0,0,12458
2,484214,0,0,0
3,181768,0,0,0
4,22453,0,0,0
5,1466,0,0,0
6,99,0,0,0
7,8,0,0,0
8,2,0,0,0


rows_per_case,cases
u32,u32
1,769339
2,273106
3,302446
4,159315
5,20987
6,1367
7,91
8,6
10,2


sex_738L,gender_992L,len
str,str,u32
"""F""","""F""",10507
"""M""","""M""",1951


num_group1,gender_992L,len
i64,str,u32
0,"""F""",10507
0,"""M""",1951
1,"""F""",10474
1,"""M""",1984


employment_start_on_15th,birth_date_on_1st,income_exactly_200000,income_non_null
f64,f64,u32,u32
0.865,1.0,12430,1526659


`person_1` lists every person on an application, with row 0 being the applicant. Sex (`sex_738L`) and birth date (`birth_259D`) are recorded once per application for all 1,526,659 applications, so the protected attributes have no missing values. Birth dates are rounded to the month and employment start dates to mid-month, and income is capped at $200,000.

In [33]:
# credit_bureau_b_1
analyze("train_credit_bureau_b_1", ["case_id", "num_group1"])
show("is year 1900 a 'never' code?", lz("train_credit_bureau_b_1").select(
    (pl.col("dpdmaxdateyear_742T") == 1900).sum().alias("dpdmax_year_1900"),
    (pl.col("overdueamountmaxdateyear_432T") == 1900).sum().alias("overdue_year_1900")).collect())


=================== train_credit_bureau_b_1: 85,791 rows, 45 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",85791,85791,0


column,pattern,rows
str,str,u32
"""contractdate_551D""","""9999-99-99""",81899
"""contractmaturitydate_151D""","""9999-99-99""",81712
"""lastupdate_260D""","""9999-99-99""",81899


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""contractdate_551D""","""2000-01-15""","""2020-10-14""",0,0,0.035,0.045
"""contractmaturitydate_151D""","""2003-04-11""","""2045-08-17""",0,651,0.034,0.048
"""lastupdate_260D""","""2015-03-30""","""2020-10-19""",0,0,0.039,0.045


column,distinct
str,i64
"""credor_3940957M""",151
"""contracttype_653M""",25
"""purposeofcred_722M""",16
"""contractst_516M""",15
"""classificationofcontr_1114M""",10
"""periodicityofpmts_997M""",10
"""pmtmethod_731M""",10
"""subjectrole_326M""",6
"""subjectrole_43M""",6


column,value,rows,share
str,str,u32,f64
"""classificationofcontr_1114M""","""ea6782cc""",67530,0.7871
"""classificationofcontr_1114M""","""01f63ac8""",7405,0.0863
"""classificationofcontr_1114M""","""00135d9c""",4794,0.0559
"""contractst_516M""","""7241344e""",74448,0.8678
"""contractst_516M""","""a55475b1""",4121,0.048
"""contractst_516M""","""8f3a197f""",1906,0.0222
"""contracttype_653M""","""4257cbed""",41626,0.4852
"""contracttype_653M""","""1c9c5356""",37508,0.4372
"""contracttype_653M""","""a55475b1""",3892,0.0454


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""amount_1115A""",0.2,4000.0,60000.0,2.54625e6,5.4833332e7,0,0.0
"""credlmt_1052A""",0.0,0.0,36184.0,1020932.6,7.968e8,0,0.259
"""credlmt_228A""",0.0,0.0,22600.0,460156.4,4.42e6,0,0.278
"""credlmt_3940954A""",0.0,0.0,20000.0,840000.0,3e8,0,0.26
"""credquantity_1099L""",1.0,1.0,1.0,5.0,16.0,0,0.0
"""credquantity_984L""",1.0,1.0,3.0,24.0,146.0,0,0.0
"""debtpastduevalue_732A""",0.0,0.0,0.0,77921.53,4.113871e7,0,0.923
"""debtvalue_227A""",0.0,0.0,35893.414,2311026.8,4.161905e7,0,0.025
"""dpd_550P""",0.0,0.0,0.0,550963.0,2.07823776e8,0,0.901


dpdmax_year_1900,overdue_year_1900
u32,u32
1,1


One row in `credit_bureau_b_1` has an overdue year of 1900 on a 2012 contract; it is a data-entry or system error.
Additionally, three "days past due" columns hold values no day count can have (up to 207 million), and the interest-rate columns reach 46,334 and 73,000.

In [34]:
analyze("train_debitcard_1", ["case_id", "num_group1"])


=================== train_debitcard_1: 157,302 rows, 6 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",157302,157302,0


column,pattern,rows
str,str,u32
"""openingdate_857D""","""9999-99-99""",144591


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""openingdate_857D""","""2001-11-19""","""2017-07-31""",0,0,0.036,0.081


--- cardinality of string/boolean columns: nothing to report
--- top values of masked (M) columns; a55475b1 is a mask for a missing value: nothing to report


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""last180dayaveragebalance_704A""",-308.79413,0.0,0.0,2425.1448,67777.77,1,0.659
"""last180dayturnover_1134A""",-187780.0,0.12,30000.0,154779.08,1.16182e6,9,0.0
"""last30dayturnover_651A""",-477.506,0.0,0.0,100000.0,390000.0,1,0.885


In [35]:
analyze("train_deposit_1", ["case_id", "num_group1"])


=================== train_deposit_1: 145,086 rows, 5 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",145086,145086,0


column,pattern,rows
str,str,u32
"""contractenddate_991D""","""9999-99-99""",65404
"""openingdate_313D""","""9999-99-99""",145086


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""contractenddate_991D""","""2002-02-27""","""2020-03-18""",0,0,0.037,0.549
"""openingdate_313D""","""2001-11-19""","""2017-07-31""",0,0,0.036,0.0


--- cardinality of string/boolean columns: nothing to report
--- top values of masked (M) columns; a55475b1 is a mask for a missing value: nothing to report


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""amount_416A""",-40000.0,0.0,223.658,178933.72,1.2213286e7,10,0.407


In [36]:
analyze("train_other_1", ["case_id", "num_group1"])


=================== train_other_1: 51,109 rows, 7 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",51109,51109,0


--- text date formats (pattern, 9 = digit): nothing to report
--- date ranges: nothing to report
--- cardinality of string/boolean columns: nothing to report
--- top values of masked (M) columns; a55475b1 is a mask for a missing value: nothing to report


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""amtdebitincoming_4809443A""",0.0,0.0,0.0,66666.6,4.957852e6,0,0.529
"""amtdebitoutgoing_4809440A""",0.0,0.0,0.0,66666.6,5168004.5,0,0.534
"""amtdepositbalance_4809441A""",-335718.0,0.0,0.0,216399.0,4256314.5,1,0.631
"""amtdepositincoming_4809444A""",0.0,0.0,0.0,48933.402,4180150.5,0,0.9
"""amtdepositoutgoing_4809442A""",0.0,0.0,1.8000001,66820.0,4622917.5,0,0.439


In [37]:
analyze("train_tax_registry_a_1", ["case_id", "num_group1"])


=================== train_tax_registry_a_1: 3,275,770 rows, 5 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",3275770,3275770,0


column,pattern,rows
str,str,u32
"""recorddate_4527225D""","""9999-99-99""",3275770


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""recorddate_4527225D""","""2019-09-13""","""2020-10-19""",0,0,0.036,0.0


column,distinct
str,i64
"""name_4527232M""",147037


column,value,rows,share
str,str,u32,f64
"""name_4527232M""","""5e180ef0""",204371,0.0624
"""name_4527232M""","""P114_118_163""",15751,0.0048
"""name_4527232M""","""74ca9587""",13196,0.004


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""amount_4527230A""",0.0,51.4,1400.0,16109.4,87115.6,0,0.0


In [38]:
analyze("train_tax_registry_b_1", ["case_id", "num_group1"])


=================== train_tax_registry_b_1: 1,107,933 rows, 5 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",1107933,1107933,0


column,pattern,rows
str,str,u32
"""deductiondate_4917603D""","""9999-99-99""",1107933


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""deductiondate_4917603D""","""2019-09-27""","""2020-10-16""",0,0,0.04,0.0


column,distinct
str,i64
"""name_4917606M""",55857


column,value,rows,share
str,str,u32,f64
"""name_4917606M""","""5e180ef0""",85284,0.077
"""name_4917606M""","""P114_118_163""",7205,0.0065
"""name_4917606M""","""74ca9587""",7153,0.0065


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""amount_4917619A""",0.0,450.0,13130.2,124740.0,344250.0,0,0.0


In [39]:
analyze("train_tax_registry_c_1", ["case_id", "num_group1"])


=================== train_tax_registry_c_1: 3,343,800 rows, 5 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1""",3343800,3343800,0


column,pattern,rows
str,str,u32
"""processingdate_168D""","""9999-99-99""",3343800


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""processingdate_168D""","""2018-07-11""","""2019-10-22""",0,0,0.031,0.0


column,distinct
str,i64
"""employername_160M""",152835


column,value,rows,share
str,str,u32,f64
"""employername_160M""","""5e180ef0""",209085,0.0625
"""employername_160M""","""6a3d9351""",18121,0.0054
"""employername_160M""","""f10df922""",14002,0.0042


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""pmtamount_36A""",0.0,40.0,1365.454,15809.359,87115.6,0,0.0


In [40]:
# tax registries: periods covered and overlap
show("applications and decision dates covered",
     pl.concat([period(f"train_tax_registry_{t}_1") for t in ("a", "b", "c")]))
ids = {t: set(lz(f"train_tax_registry_{t}_1").select("case_id").unique().collect()["case_id"].to_list())
       for t in ("a", "b", "c")}
show("applications in both registries", pl.DataFrame({
    "pair": ["a and b", "a and c", "b and c"],
    "applications": [len(ids["a"] & ids["b"]), len(ids["a"] & ids["c"]), len(ids["b"] & ids["c"])]}))

table,applications,first_decision,last_decision
str,u32,date,date
"""train_tax_registry_a_1""",457934,2019-08-30,2020-10-05
"""train_tax_registry_b_1""",150732,2020-03-12,2020-10-05
"""train_tax_registry_c_1""",482265,2019-01-01,2019-10-14


pair,applications
str,i64
"""a and b""",4246
"""a and c""",82920
"""b and c""",0


The three tax registries cover successive periods: `tax_registry_c_1` for applications decided January to mid-October 2019, tax_registry_a_1 from late August 2019 to October 2020, and tax_registry_b_1 from mid-March to October 2020. They overlap only at the handovers, and together about two-thirds of applications have a tax record in every bracket, so missingness in any single registry mainly reflects the application date.

## Depth 2: several rows per (case, person) (key = case_id, num_group1, num_group2)

In [41]:
analyze("train_person_2", ["case_id", "num_group1", "num_group2"])


=================== train_person_2: 1,643,410 rows, 11 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1, num_group2""",1643410,1643410,0


column,pattern,rows
str,str,u32
"""empls_employedfrom_796D""","""9999-99-99""",5757


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,f64,f64
"""empls_employedfrom_796D""","""1972-08-15""","""2020-01-19""",0,0,0.002,0.996


column,distinct
str,i64
"""empls_employer_name_740M""",7153
"""addres_zip_823M""",2027
"""addres_district_368M""",508
"""conts_role_79M""",11
"""empls_economicalst_849M""",10
"""relatedpersons_role_762T""",10
"""addres_role_871L""",8


column,value,rows,share
str,str,u32,f64
"""addres_district_368M""","""a55475b1""",1582872,0.9632
"""addres_district_368M""","""P125_48_164""",9669,0.0059
"""addres_district_368M""","""P155_139_77""",4093,0.0025
"""addres_zip_823M""","""a55475b1""",1576370,0.9592
"""addres_zip_823M""","""P161_14_174""",5968,0.0036
"""addres_zip_823M""","""P144_138_111""",3405,0.0021
"""conts_role_79M""","""a55475b1""",1587829,0.9662
"""conts_role_79M""","""P38_92_157""",29333,0.0178
"""conts_role_79M""","""P177_137_98""",9179,0.0056


--- numeric ranges: nothing to report


In [42]:
analyze("train_credit_bureau_b_2", ["case_id", "num_group1", "num_group2"])


=================== train_credit_bureau_b_2: 1,286,755 rows, 6 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1, num_group2""",1286755,1286755,0


column,pattern,rows
str,str,u32
"""pmts_date_1107D""","""9999-99-99""",1286755


column,min,max,before_1930,after_2030,share_day_is_1,null_rate
str,str,str,i64,i64,i64,f64
"""pmts_date_1107D""","""2016-01-15""","""2020-10-15""",0,0,0,0.0


--- cardinality of string/boolean columns: nothing to report
--- top values of masked (M) columns; a55475b1 is a mask for a missing value: nothing to report


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""pmts_dpdvalue_108P""",0.0,0.0,0.0,577145.0,1.85124192e8,0,0.887
"""pmts_pmtsoverdue_635A""",0.0,0.0,0.0,319.80002,147470.61,0,0.887


In [43]:
analyze("train_credit_bureau_a_2_7", ["case_id", "num_group1", "num_group2"])


=================== train_credit_bureau_a_2_7: 8,055,986 rows, 19 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1, num_group2""",8055986,8055986,0


--- text date formats (pattern, 9 = digit): nothing to report
--- date ranges: nothing to report


column,distinct
str,i64
"""collaterals_typeofguarante_359M""",15
"""collaterals_typeofguarante_669M""",15
"""subjectroles_name_541M""",10
"""subjectroles_name_838M""",7
"""collater_typofvalofguarant_407M""",6
"""collater_typofvalofguarant_298M""",4


column,value,rows,share
str,str,u32,f64
"""collater_typofvalofguarant_298M""","""a55475b1""",7956618,0.9877
"""collater_typofvalofguarant_298M""","""9a0c095e""",61356,0.0076
"""collater_typofvalofguarant_298M""","""8fd95e4b""",37968,0.0047
"""collater_typofvalofguarant_407M""","""a55475b1""",7720433,0.9583
"""collater_typofvalofguarant_407M""","""9a0c095e""",172448,0.0214
"""collater_typofvalofguarant_407M""","""8fd95e4b""",162673,0.0202
"""collaterals_typeofguarante_359M""","""a55475b1""",7720433,0.9583
"""collaterals_typeofguarante_359M""","""c7a5ad39""",251636,0.0312
"""collaterals_typeofguarante_359M""","""3cbe86ba""",47608,0.0059


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""collater_valueofguarantee_1124L""",0.0,0.0,0.0,8.4744e7,1.7500e9,0,0.912
"""collater_valueofguarantee_876L""",0.0,0.0,0.0,2.75557e8,1.5769e10,0,0.876
"""pmts_dpd_1073P""",0.0,0.0,0.0,197.0,4520.0,0,0.942
"""pmts_dpd_303P""",-12.0,0.0,0.0,1399.0,84574.0,531,0.837
"""pmts_month_158T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_month_706T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_overdue_1140A""",0.0,0.0,0.0,17786.65,1.4993038e8,0,0.941
"""pmts_overdue_1152A""",0.0,0.0,0.0,74622.4,4.6413904e7,0,0.83
"""pmts_year_1139T""",2016.0,2018.0,2019.0,2021.0,2021.0,0,0.0


In [44]:
analyze("train_credit_bureau_a_2_8", ["case_id", "num_group1", "num_group2"])


=================== train_credit_bureau_a_2_8: 13,927,071 rows, 19 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1, num_group2""",13927071,13927071,0


--- text date formats (pattern, 9 = digit): nothing to report
--- date ranges: nothing to report


column,distinct
str,i64
"""collaterals_typeofguarante_359M""",15
"""collaterals_typeofguarante_669M""",14
"""subjectroles_name_541M""",10
"""collater_typofvalofguarant_407M""",7
"""subjectroles_name_838M""",7
"""collater_typofvalofguarant_298M""",4


column,value,rows,share
str,str,u32,f64
"""collater_typofvalofguarant_298M""","""a55475b1""",13778078,0.9893
"""collater_typofvalofguarant_298M""","""9a0c095e""",99894,0.0072
"""collater_typofvalofguarant_298M""","""8fd95e4b""",49026,0.0035
"""collater_typofvalofguarant_407M""","""a55475b1""",13373863,0.9603
"""collater_typofvalofguarant_407M""","""9a0c095e""",299054,0.0215
"""collater_typofvalofguarant_407M""","""8fd95e4b""",253478,0.0182
"""collaterals_typeofguarante_359M""","""a55475b1""",13373863,0.9603
"""collaterals_typeofguarante_359M""","""c7a5ad39""",437459,0.0314
"""collaterals_typeofguarante_359M""","""3cbe86ba""",80058,0.0057


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""collater_valueofguarantee_1124L""",0.0,0.0,0.0,1.7277532e7,3.2000e9,0,0.935
"""collater_valueofguarantee_876L""",0.0,0.0,0.0,4.877329e6,6.8050e9,0,0.912
"""pmts_dpd_1073P""",0.0,0.0,0.0,18.0,4520.0,0,0.958
"""pmts_dpd_303P""",-9.0,0.0,0.0,1454.0,84575.0,779,0.839
"""pmts_month_158T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_month_706T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_overdue_1140A""",0.0,0.0,0.0,7409.2,5881917.5,0,0.958
"""pmts_overdue_1152A""",0.0,0.0,0.0,71066.4,5.1793576e7,0,0.832
"""pmts_year_1139T""",2016.0,2018.0,2019.0,2021.0,2021.0,0,0.0


In [45]:
analyze("train_credit_bureau_a_2_9", ["case_id", "num_group1", "num_group2"])


=================== train_credit_bureau_a_2_9: 18,723,227 rows, 19 columns ===================


keys,rows,unique_keys,duplicate_rows
str,i64,i64,i64
"""case_id, num_group1, num_group2""",18723227,18723227,0


--- text date formats (pattern, 9 = digit): nothing to report
--- date ranges: nothing to report


column,distinct
str,i64
"""collaterals_typeofguarante_359M""",15
"""collaterals_typeofguarante_669M""",14
"""subjectroles_name_541M""",9
"""collater_typofvalofguarant_407M""",7
"""subjectroles_name_838M""",7
"""collater_typofvalofguarant_298M""",5


column,value,rows,share
str,str,u32,f64
"""collater_typofvalofguarant_298M""","""a55475b1""",18526243,0.9895
"""collater_typofvalofguarant_298M""","""9a0c095e""",133066,0.0071
"""collater_typofvalofguarant_298M""","""8fd95e4b""",63825,0.0034
"""collater_typofvalofguarant_407M""","""a55475b1""",17972743,0.9599
"""collater_typofvalofguarant_407M""","""9a0c095e""",404065,0.0216
"""collater_typofvalofguarant_407M""","""8fd95e4b""",345159,0.0184
"""collaterals_typeofguarante_359M""","""a55475b1""",17973072,0.9599
"""collaterals_typeofguarante_359M""","""c7a5ad39""",594340,0.0317
"""collaterals_typeofguarante_359M""","""3cbe86ba""",105770,0.0056


column,min,p1,median,p99,max,n_negative,share_zero
str,f64,f64,f64,f64,f64,i64,f64
"""collater_valueofguarantee_1124L""",0.0,0.0,0.0,7.9934932e7,6.0272e9,0,0.922
"""collater_valueofguarantee_876L""",0.0,0.0,0.0,4.557649e6,5.5000e9,0,0.912
"""pmts_dpd_1073P""",0.0,0.0,0.0,25.0,4212.0,0,0.954
"""pmts_dpd_303P""",-30.0,0.0,0.0,1426.0,84575.0,1057,0.843
"""pmts_month_158T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_month_706T""",1.0,1.0,6.5,12.0,12.0,0,0.0
"""pmts_overdue_1140A""",0.0,0.0,0.0,8740.61,4.373397e8,0,0.953
"""pmts_overdue_1152A""",0.0,0.0,0.0,70722.43,1.2069077e8,0,0.837
"""pmts_year_1139T""",2016.0,2018.0,2019.0,2021.0,2021.0,0,0.0


In [47]:
# credit_bureau_a_2: payment months after the decision date (possible leakage)
# pmts_year_* / pmts_month_* are T columns, so the date checks above do not see them.
a2 = pl.concat([lz(f"train_credit_bureau_a_2_{s}").select("case_id", "pmts_year_1139T", "pmts_month_158T",
                                                          "pmts_dpd_1073P", "pmts_overdue_1140A") for s in "789"])
pay_month = pl.col("pmts_year_1139T").cast(pl.Int64) * 12 + pl.col("pmts_month_158T").cast(pl.Int64)
dec_month = pl.col("decision").dt.year().cast(pl.Int64) * 12 + pl.col("decision").dt.month().cast(pl.Int64)
timing = (a2.filter(pl.col("pmts_year_1139T").is_not_null()).join(DECISION, on="case_id")
          .with_columns(pl.when(pay_month > dec_month).then(pl.lit("3 after decision month"))
                        .when(pay_month == dec_month).then(pl.lit("2 decision month"))
                        .otherwise(pl.lit("1 before decision month")).alias("when")))
show("active-contract payment rows by month relative to the decision", timing.group_by("when").agg(
    pl.len().alias("rows"), pl.col("case_id").n_unique().alias("applications"),
    pl.col("pmts_dpd_1073P").is_null().mean().round(3).alias("dpd_null"),
    (pl.col("pmts_dpd_1073P") > 0).sum().alias("rows_dpd_positive"),
    (pl.col("pmts_overdue_1140A") > 0).sum().alias("rows_overdue_positive"),
    (pay_month - dec_month).max().alias("max_months_after")).sort("when").collect())
show("rows after the decision month, by months after", timing.filter(pay_month > dec_month)
     .group_by((pay_month - dec_month).alias("months_after")).agg(
    pl.len().alias("rows"), pl.col("case_id").n_unique().alias("applications"),
    pl.col("pmts_dpd_1073P").is_not_null().sum().alias("rows_dpd_filled"),
    (pl.col("pmts_dpd_1073P") > 0).sum().alias("rows_dpd_positive"),
    pl.col("case_id").filter(pl.col("pmts_dpd_1073P") > 0).n_unique().alias("applications_dpd_positive"))
     .sort("months_after").collect())

when,rows,applications,dpd_null,rows_dpd_positive,rows_overdue_positive,max_months_after
str,u32,u32,f64,u32,u32,i64
"""1 before decision month""",8001566,197355,0.332,259881,264965,-1
"""2 decision month""",426223,197342,0.022,17724,18065,0
"""3 after decision month""",2746227,197342,0.901,6161,6288,10


months_after,rows,applications,rows_dpd_filled,rows_dpd_positive,applications_dpd_positive
i64,u32,u32,u32,u32,u32
1,426223,197342,270869,6161,5335
2,426223,197342,0,0,0
3,426223,197342,0,0,0
4,426223,197342,0,0,0
5,344784,158989,0,0,0
6,255305,116506,0,0,0
7,202074,92032,0,0,0
8,119147,53660,0,0,0
9,67447,29546,0,0,0


In `credit_bureau_a_2` (files 7–9), the monthly payment grid extends up to 10 months past the decision month, but only the first month after the decision has values: 270,869 of its 426,223 rows have a days-past-due value, almost all of them 0, and 6,161 show a payment past due. Months 2–10 are empty. This may be data recorded after the decision or a month label running ahead of the calendar. In that first month, 6,161 rows (5,335 applications) show days past due, compared with 259,881 overdue rows before the decision month and 17,724 in it, so about 2% of the overdue rows the audited code aggregates come from after the decision.

## Possible cleaning approaches

`work/clean/` is a separate typed copy of the same casts the audited code applies.

After that type casting, the only other cleaning the audited code does is:
- Drop a column if it is over 95% null.
- Drop a string column with exactly 1 distinct value or > 50 distinct values
- Drop a column whose name contains `month_ `or `year_` without a suffix of P, A, L, or M.
- Four `max - min` features fill nulls with 0 before the subtraction (`amount_4527230A`, `collater_valueofguarantee_1124L`, `pmts_dpd_1073P` and, `pmts_overdue_1140A`).
- `D` columns become days since `date_decision` (`handle_dates`). A null date stays null. A date far
  outside a plausible range stays a large day count.

Further cleaning rules our audit can apply:
- blank strings (after trimming) become null
- `placeholders={"col": [value, ...]}`: values that mean "missing" become null
  - `a55475b1` and `risk_assesment_940T=0`
- `date_bounds={"col": ("1930-01-01", "2005-12-31")}`: dates outside the range become null
  - `dateofbirth_337D`: 65 dates outside 1930-01-01..2005-12-31, including 1900-01-01. A sentinel date becomes "missing" instead of a day offset the audited code can split on. Age for the audit comes from `birth_259D`.
- `value_bounds={"col": (lo, hi)}`: numbers outside [lo, hi] become null (None = no bound on that side). `dpd_733P`, `pmtdaysoverdue_1135P`, `pmts_dpd_303P` all have values over 100 years.
- `derived={"new_col": expr}`: A numeric copy of an ordered code. `empl_employedtotal_800L`, `riskassesment_302T`.

# Null value trends across different attributes

In [48]:
# applicant row: employment nulls by income type, age, and sex
# Recomputes the tables in the section above. Applicant row only (num_group1 = 0).
# Age is years from birth_259D to date_decision. Reads work/clean and the raw train_base dates.
applicants = (
    lz("train_person_1").filter(pl.col("num_group1") == 0)
    .select(pl.col("case_id").cast(pl.Int64), "sex_738L", "incometype_1044T", "birth_259D",
            "empl_employedtotal_800L", "familystate_447L", "mainoccupationinc_384A")
    .join(pl.scan_parquet(RAW / "train_base.parquet").select(
        pl.col("case_id").cast(pl.Int64),
        pl.col("date_decision").str.to_date(strict=False)), on="case_id")
    .with_columns(((pl.col("date_decision") - pl.col("birth_259D")).dt.total_days() / 365.25).alias("age"))
    .with_columns(
        pl.when(pl.col("age") < 25).then(pl.lit("under 25"))
        .when(pl.col("age") < 40).then(pl.lit("25-39"))
        .when(pl.col("age") < 60).then(pl.lit("40-59"))
        .otherwise(pl.lit("60+")).alias("age_band"),
        pl.col("empl_employedtotal_800L").is_null().alias("empl_null"))
    .collect())

show("employment length null by income type",
     applicants.group_by("incometype_1044T").agg(
         pl.len().alias("applicants"),
         pl.col("empl_null").mean().round(3).alias("empl_null_rate"))
     .sort("applicants", descending=True))
show("employment length null by age at date_decision",
     applicants.group_by("age_band").agg(
         pl.len().alias("applicants"),
         pl.col("empl_null").mean().round(3).alias("empl_null_rate"))
     .sort("age_band"))
show("employment length null by sex and income type",
     applicants.group_by("incometype_1044T", "sex_738L").agg(
         pl.len().alias("applicants"),
         pl.col("empl_null").mean().round(3).alias("empl_null_rate"))
     .sort("incometype_1044T", "sex_738L"))
retired = ["RETIRED_PENSIONER", "HANDICAPPED", "HANDICAPPED_2", "HANDICAPPED_3"]
employed = ["PRIVATE_SECTOR_EMPLOYEE", "SALARIED_GOVT", "EMPLOYED", "SELFEMPLOYED"]
show("share retired or a disability income type, by sex",
     applicants.group_by("sex_738L").agg(
         pl.len().alias("applicants"),
         pl.col("incometype_1044T").is_in(retired).mean().round(3).alias("share_retired_or_disability"),
         pl.col("empl_null").mean().round(3).alias("empl_null_rate")))
show("employment length null by sex, four employed income types only",
     applicants.filter(pl.col("incometype_1044T").is_in(employed)).group_by("sex_738L").agg(
         pl.len().alias("applicants"),
         pl.col("empl_null").mean().round(3).alias("empl_null_rate")))
show("family state",
     applicants.group_by("familystate_447L").agg(pl.len().alias("applicants")).sort("applicants", descending=True))
print("income amount null rate", round(applicants.select(pl.col("mainoccupationinc_384A").is_null().mean()).item(), 3))

incometype_1044T,applicants,empl_null_rate
str,u32,f64
"""PRIVATE_SECTOR_EMPLOYEE""",490562,0.528
"""SALARIED_GOVT""",373646,0.613
"""RETIRED_PENSIONER""",311028,0.989
"""EMPLOYED""",298158,0.563
"""SELFEMPLOYED""",29199,0.498
"""OTHER""",11436,0.694
"""HANDICAPPED_2""",7371,0.986
"""HANDICAPPED_3""",5258,0.975
"""HANDICAPPED""",1,1.0


age_band,applicants,empl_null_rate
str,u32,f64
"""25-39""",564874,0.555
"""40-59""",579927,0.656
"""60+""",286374,0.967
"""under 25""",95484,0.291


incometype_1044T,sex_738L,applicants,empl_null_rate
str,str,u32,f64
"""EMPLOYED""","""F""",152988,0.571
"""EMPLOYED""","""M""",145170,0.554
"""HANDICAPPED""","""F""",1,1.0
"""HANDICAPPED_2""","""F""",3876,0.988
"""HANDICAPPED_2""","""M""",3495,0.983
"""HANDICAPPED_3""","""F""",2708,0.979
"""HANDICAPPED_3""","""M""",2550,0.971
"""OTHER""","""F""",1676,0.701
"""OTHER""","""M""",9760,0.693


sex_738L,applicants,share_retired_or_disability,empl_null_rate
str,u32,f64,f64
"""F""",952776,0.259,0.682
"""M""",573883,0.133,0.607


sex_738L,applicants,empl_null_rate
str,u32,f64
"""M""",487538,0.546
"""F""",704027,0.574


familystate_447L,applicants
str,u32
null,798046
"""MARRIED""",484846
"""SINGLE""",183334
"""WIDOWED""",32995
"""DIVORCED""",19296
"""LIVING_WITH_PARTNER""",8142


income amount null rate 0.0


## How should we handle nulls

Our hypothesis is to leave the nulls as is in some columns and set them to 0 in others.

**Why not impute the median?** Most of the nulls in the dataset are structural - they are null for a reason. For example, if we were to impute the median for all the null employment_length values in `RETIRED_PENSIONER` and `HANDICAPPED` columns, it would give a value to people with no job. Any median imputation would invent a value.

**Why not drop all nulls?**. Because an applicant with any null is nearly every applicant.

**Which columns will we set nulls to 0:** We will set nulls to 0 in placeholder codes. `riskassesment_940T=0` is a placeholder in the top risk band of the column, so setting those values to null will keep the min from being 0 and prevent skewed data. We'll also clip or null out-of-range numbers. `dpd_733P`, `pmtdaysoverdue_1135P`, and `pmts_dpd_303P` have 12, 4, and 15 values over 100 years. The audited code keeps them, and one extreme sets the max or the min in its aggregations.

In amount columns, a zero means something different than null. For example, a missing tax or debit-card row should not be set to 0, but rather kept as null. The audited code fills null with 0 only in four `max - min` features, and that pull of the minimum toward 0 is part of the system being audited.

